# TrackMate LoG optimization viewer

`outputs/TMoptimization` のLoG比較結果を、前処理済み第80スライスに重ねて確認するviewerです。

- Dataset: MAY06 / 08 / 18 / 25 × L/R × ventral/dorsal
- Condition: Radius 1.5 / 2.0 / 2.5 / 3.0 × Median OFF/ON
- 複数条件をチェックすると色分けして同時表示
- 各条件のQ percentile以上から、再現可能なランダム標本を表示
- Backgroundは前処理画像とRaw画像を切替可能

初期表示では半径2.0のMedian OFF/ONについて、各条件のQ上位10%からランダムに10,000点を描画します。条件選択後の初回XML読込には数秒かかる場合があります。

In [1]:
from functools import lru_cache
from pathlib import Path
import re
import xml.etree.ElementTree as ET
import zlib

import ipywidgets as widgets
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Project root containing src/ was not found.')

OPT_ROOT = PROJECT_ROOT / 'outputs' / 'TMoptimization'
SUMMARY_PATH = OPT_ROOT / 'optimization_summary.csv'
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(SUMMARY_PATH)

SUMMARY = pd.read_csv(str(SUMMARY_PATH))
SUMMARY['median_filter'] = SUMMARY['median_filter'].astype(str).str.lower().eq('true')
if len(SUMMARY) != 128:
    raise RuntimeError('Expected 128 optimization rows, found {}'.format(len(SUMMARY)))

DATASETS = sorted(SUMMARY['dataset'].unique())
RADII = sorted(float(value) for value in SUMMARY['radius'].unique())
MEDIAN_STATES = [False, True]
if len(DATASETS) != 16 or RADII != [1.5, 2.0, 2.5, 3.0]:
    raise RuntimeError('Unexpected dataset or radius grid.')

ROW_INDEX = {}
for _, row in SUMMARY.iterrows():
    key = (row['dataset'], float(row['radius']), bool(row['median_filter']))
    if key in ROW_INDEX:
        raise RuntimeError('Duplicate optimization row: {}'.format(key))
    ROW_INDEX[key] = row

COLORS = {
    (1.5, False): '#00bfff', (1.5, True): '#004cff',
    (2.0, False): '#00d45a', (2.0, True): '#00802b',
    (2.5, False): '#ffb000', (2.5, True): '#ff6500',
    (3.0, False): '#ff4da6', (3.0, True): '#a00070',
}

def current_path(path_string):
    path = Path(path_string)
    parts = list(path.parts)
    if 'outputs' in parts:
        index = parts.index('outputs')
        return PROJECT_ROOT.joinpath(*parts[index:])
    return path

def row_for(dataset, radius, median_filter):
    return ROW_INDEX[(dataset, float(radius), bool(median_filter))]

def processed_path(dataset):
    return current_path(row_for(dataset, RADII[0], False)['input_tif'])

def raw_path(dataset):
    processed = processed_path(dataset)
    stem = re.sub(r'_\d{3}$', '', processed.stem)
    return PROJECT_ROOT / 'data' / (stem + '.tif')

def xml_path(dataset, radius, median_filter):
    stored = Path(row_for(dataset, radius, median_filter)['output_xml'])
    return OPT_ROOT / 'raw_xml' / dataset / stored.name

for dataset in DATASETS:
    if not processed_path(dataset).is_file():
        raise FileNotFoundError(processed_path(dataset))
    if not raw_path(dataset).is_file():
        raise FileNotFoundError(raw_path(dataset))
    for radius in RADII:
        for median_filter in MEDIAN_STATES:
            if not xml_path(dataset, radius, median_filter).is_file():
                raise FileNotFoundError(xml_path(dataset, radius, median_filter))

print('Project:', PROJECT_ROOT)
print('Optimization datasets:', len(DATASETS))
print('Optimization XML files:', len(SUMMARY))

Project: C:\workspace\LSFM_pp
Optimization datasets: 16
Optimization XML files: 128


In [2]:
@lru_cache(maxsize=8)
def read_spots(xml_path_string):
    count = None
    spots = None
    index = 0
    for event, element in ET.iterparse(xml_path_string, events=('start', 'end')):
        if event == 'start' and element.tag == 'AllSpots':
            count = int(element.attrib['nspots'])
            spots = np.empty((count, 3), dtype=np.float32)
        elif event == 'end' and element.tag == 'Spot':
            spots[index, 0] = float(element.attrib['POSITION_X'])
            spots[index, 1] = float(element.attrib['POSITION_Y'])
            spots[index, 2] = float(element.attrib['QUALITY'])
            index += 1
        if event == 'end':
            element.clear()
    if spots is None or index != count:
        raise RuntimeError('Spot count mismatch in {}'.format(xml_path_string))
    return spots

@lru_cache(maxsize=4)
def read_processed_image(path_string):
    return tiff.imread(path_string)

@lru_cache(maxsize=4)
def read_raw_image(path_string, page_index=79):
    with tiff.TiffFile(path_string) as tif_file:
        return tif_file.pages[int(page_index)].asarray()

def read_background(dataset, source):
    if source == 'processed':
        return read_processed_image(str(processed_path(dataset).resolve()))
    return read_raw_image(str(raw_path(dataset).resolve()), 79)

def random_sample_spots(spots, quality_percentile, sample_n, seed):
    cutoff = float(np.percentile(spots[:, 2], float(quality_percentile)))
    eligible = np.flatnonzero(spots[:, 2] >= cutoff)
    eligible_count = len(eligible)
    if eligible_count > int(sample_n):
        rng = np.random.RandomState(int(seed) & 0xffffffff)
        eligible = rng.choice(eligible, size=int(sample_n), replace=False)
    return spots[eligible], eligible_count, cutoff

def clipped_view_bounds(center, full_size, zoom):
    view_size = float(full_size) / float(zoom)
    lower = float(center) - view_size / 2.0
    lower = min(max(lower, 0.0), max(float(full_size) - view_size, 0.0))
    return lower, lower + view_size

In [3]:
dataset_widget = widgets.Dropdown(
    options=DATASETS, value=DATASETS[0], description='Dataset:',
    layout=widgets.Layout(width='55%')
)
background_widget = widgets.ToggleButtons(
    options=[('Preprocessed', 'processed'), ('Raw', 'raw')],
    value='processed', description='Background:'
)

condition_widgets = {}
condition_rows = []
for radius in RADII:
    row_widgets = []
    for median_filter in MEDIAN_STATES:
        label = 'R={:.1f} Median {}'.format(radius, 'ON' if median_filter else 'OFF')
        checkbox = widgets.Checkbox(
            value=(radius == 2.0), description=label, indent=False,
            layout=widgets.Layout(width='190px')
        )
        condition_widgets[(radius, median_filter)] = checkbox
        row_widgets.append(checkbox)
    condition_rows.append(widgets.HBox(row_widgets))

quality_percentile_widget = widgets.FloatSlider(
    value=90.0, min=0.0, max=99.9, step=0.5, description='Q percentile:',
    continuous_update=False, readout_format='.1f', layout=widgets.Layout(width='55%')
)
sample_n_widget = widgets.BoundedIntText(
    value=10000, min=100, max=100000, step=100, description='Random N:'
)
marker_size_widget = widgets.FloatSlider(
    value=4.0, min=1.0, max=20.0, step=1.0, description='Marker:',
    continuous_update=False, layout=widgets.Layout(width='45%')
)
marker_alpha_widget = widgets.FloatSlider(
    value=0.7, min=0.1, max=1.0, step=0.1, description='Alpha:',
    continuous_update=False, layout=widgets.Layout(width='45%')
)
contrast_widget = widgets.FloatRangeSlider(
    value=(0.5, 99.7), min=0.0, max=100.0, step=0.1,
    description='Percentile:', continuous_update=False,
    readout_format='.1f', layout=widgets.Layout(width='75%')
)
zoom_widget = widgets.SelectionSlider(
    options=[1, 1.5, 2, 3, 4, 6, 8, 12, 16], value=1,
    description='Zoom:', continuous_update=False,
    layout=widgets.Layout(width='45%')
)
center_x_widget = widgets.IntSlider(
    value=2048, min=0, max=4095, step=1, description='Center X:',
    continuous_update=False, layout=widgets.Layout(width='80%')
)
center_y_widget = widgets.IntSlider(
    value=1080, min=0, max=2159, step=1, description='Center Y:',
    continuous_update=False, layout=widgets.Layout(width='80%')
)
figure_width_widget = widgets.FloatSlider(
    value=11.0, min=5.0, max=16.0, step=0.5, description='Figure:',
    continuous_update=False, layout=widgets.Layout(width='45%')
)
reset_view_button = widgets.Button(description='Reset view', icon='refresh')
pan_left_button = widgets.Button(description='←', layout=widgets.Layout(width='45px'))
pan_right_button = widgets.Button(description='→', layout=widgets.Layout(width='45px'))
pan_up_button = widgets.Button(description='↑', layout=widgets.Layout(width='45px'))
pan_down_button = widgets.Button(description='↓', layout=widgets.Layout(width='45px'))
refresh_button = widgets.Button(description='Refresh', icon='play')
status_widget = widgets.HTML()
viewer_output = widgets.Output()
_updating = False

def selected_conditions():
    return [key for key, checkbox in condition_widgets.items() if checkbox.value]

def reset_view(_=None, render=True):
    global _updating
    image = read_background(dataset_widget.value, background_widget.value)
    height, width = image.shape
    _updating = True
    center_x_widget.max = width - 1
    center_y_widget.max = height - 1
    center_x_widget.value = width // 2
    center_y_widget.value = height // 2
    zoom_widget.value = 1
    _updating = False
    if render:
        render_view()

def pan_view(dx, dy):
    global _updating
    image = read_background(dataset_widget.value, background_widget.value)
    height, width = image.shape
    zoom = float(zoom_widget.value)
    step_x = max(1, int((width / zoom) * 0.25))
    step_y = max(1, int((height / zoom) * 0.25))
    _updating = True
    center_x_widget.value = int(np.clip(center_x_widget.value + dx * step_x, 0, width - 1))
    center_y_widget.value = int(np.clip(center_y_widget.value + dy * step_y, 0, height - 1))
    _updating = False
    render_view()

def render_view(_=None):
    if _updating:
        return
    dataset = dataset_widget.value
    conditions = selected_conditions()
    if not conditions:
        status_widget.value = '<b style="color:red">Select at least one condition.</b>'
        return
    try:
        image = read_background(dataset, background_widget.value)
        low_p, high_p = contrast_widget.value
        vmin, vmax = np.percentile(image, [low_p, high_p])
        if vmax <= vmin:
            vmax = vmin + 1
        height, width = image.shape
        zoom = float(zoom_widget.value)
        x0, x1 = clipped_view_bounds(center_x_widget.value, width, zoom)
        y0, y1 = clipped_view_bounds(center_y_widget.value, height, zoom)
        figure_width = float(figure_width_widget.value)
        figure_height = min(11.0, max(3.0, figure_width * (y1 - y0) / max(x1 - x0, 1)))
        fig, ax = plt.subplots(figsize=(figure_width, figure_height), dpi=120)
        ax.imshow(image, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')

        summary_lines = []
        for radius, median_filter in conditions:
            path = xml_path(dataset, radius, median_filter)
            spots = read_spots(str(path.resolve()))
            seed_text = '{}|{}|{}|{}'.format(
                dataset, radius, median_filter, quality_percentile_widget.value
            )
            seed = zlib.crc32(seed_text.encode('utf-8')) & 0xffffffff
            shown, eligible_count, q_cutoff = random_sample_spots(
                spots, quality_percentile_widget.value, sample_n_widget.value, seed
            )
            label = 'R={:.1f} median {} ({:,} shown / {:,} eligible; Q≥{:.2f})'.format(
                radius, 'ON' if median_filter else 'OFF', len(shown), eligible_count, q_cutoff
            )
            ax.scatter(
                shown[:, 0], shown[:, 1], s=marker_size_widget.value,
                facecolors='none', edgecolors=COLORS[(radius, median_filter)],
                linewidths=0.7, alpha=marker_alpha_widget.value, label=label,
            )
            summary_lines.append(label)

        ax.set_xlim(x0, x1)
        ax.set_ylim(y1, y0)
        ax.set_title('{} | slice 080 | {} | zoom {}x'.format(
            dataset, background_widget.label, zoom_widget.value
        ))
        ax.set_xlabel('X (pixel)')
        ax.set_ylabel('Y (pixel)')
        ax.legend(loc='upper right', fontsize=8, framealpha=0.8)
        fig.tight_layout()

        status_widget.value = (
            '<b>{}</b> | {} condition(s) | Q percentile ≥ {:.1f} | random N={} per condition<br>{}'.format(
                dataset, len(conditions), quality_percentile_widget.value,
                sample_n_widget.value, '<br>'.join(summary_lines)
            )
        )
        with viewer_output:
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
    except Exception as error:
        status_widget.value = '<b style="color:red">{}: {}</b>'.format(type(error).__name__, error)
        with viewer_output:
            clear_output(wait=True)
            print(type(error).__name__ + ':', error)

def change_dataset(_=None):
    reset_view(render=False)
    render_view()

reset_view_button.on_click(reset_view)
pan_left_button.on_click(lambda _: pan_view(-1, 0))
pan_right_button.on_click(lambda _: pan_view(1, 0))
pan_up_button.on_click(lambda _: pan_view(0, -1))
pan_down_button.on_click(lambda _: pan_view(0, 1))
refresh_button.on_click(render_view)
dataset_widget.observe(change_dataset, names='value')
background_widget.observe(change_dataset, names='value')
for checkbox in condition_widgets.values():
    checkbox.observe(render_view, names='value')
for widget in [
    quality_percentile_widget, sample_n_widget, marker_size_widget, marker_alpha_widget,
    contrast_widget, zoom_widget, center_x_widget, center_y_widget, figure_width_widget,
]:
    widget.observe(render_view, names='value')

controls = widgets.VBox([
    widgets.HBox([dataset_widget, background_widget]),
    widgets.HTML('<b>Conditions</b>'),
    widgets.VBox(condition_rows),
    widgets.HBox([quality_percentile_widget, sample_n_widget, refresh_button]),
    widgets.HBox([marker_size_widget, marker_alpha_widget]),
    contrast_widget,
    widgets.HBox([zoom_widget, figure_width_widget, reset_view_button]),
    center_x_widget,
    center_y_widget,
    widgets.HBox([pan_left_button, pan_right_button, pan_up_button, pan_down_button]),
    status_widget,
])

reset_view(render=False)
display(controls, viewer_output)
render_view()

Output()